# Small surface defect detection - end-to-end pipeline

One notebook for the whole study (plan: `docs/KE_HOACH_NGHIEN_CUU_LOI_BE_MAT_NHO.md`):
data -> EDA -> phase 1 baselines -> phase 2 screening / large images / ablation -> phase 3 transfer -> tables & figures.

**How to use on Kaggle**
1. Settings: **Accelerator = GPU T4 x2** (T4 x1 also works), **Internet = on**.
2. Add Data: the raw datasets (slugs in `configs/paths.yaml`), the code (GitHub or a dataset `sdd-code`), and -
   from the 2nd session on - the output of the previous version of this notebook.
3. Pick the stages in the **Config** cell, then **Save Version -> Save & Run All**.

The full study does not fit in one 12 h session. Every run is resumable: finished runs are skipped, interrupted runs
continue from `last.pt`. Set `PREV_RESULTS` to the previous version's output and run again. The whole notebook
shares one time budget (`SESSION_HOURS`), so it stops launching new runs before Kaggle ends the session.

## 1. Config

In [ ]:
import os, time
T0 = time.time()

# --- code & data ---------------------------------------------------------------------------------------
SDD_REPO       = "https://github.com/Dlevinh755/small-defect-detection.git"  # "" -> use SDD_CODE_INPUT
SDD_CODE_INPUT = "/kaggle/input/sdd-code"            # used when SDD_REPO is empty
PREV_RESULTS   = ""                                  # e.g. "/kaggle/input/sdd-e2e-v3/results" (resume)

SOURCE_DATASETS = ["neu", "gc10", "pcb", "mt"]
TARGET_DATASET  = "ksdd2"

# --- stages (all resumable; re-running a finished stage only re-reads its results) ---------------------
SMOKE = False   # True: every model 1 epoch on 10% of NEU - pipeline check only (~15 min)
RUN = {
    "data":        True,   # convert datasets + draw labels (check them before training!)
    "eda":         True,
    "p1":          True,   # baselines + P2 (plan §4)
    "p2_screen":   False,  # single modules, 1 seed -> pick A2/A3 in configs/variants.yaml (plan §5.2)
    "p2_tiling":   False,  # PCB / GC10: SAHI on the P1 checkpoints + training on 640 tiles
    "p2_ablation": False,  # A0-A7 x 3 seeds (edit variants.yaml first!)
    "p3":          False,  # source models + transfer grid on the target set (plan §6)
    "report":      True,   # tables / figures / feature maps / error galleries
}
SESSION_HOURS = 11.5       # Kaggle limit is 12 h; margin for the report + saving the output

## 2. Setup

In [ ]:
os.environ.update(SDD_REPO=SDD_REPO, SDD_CODE_INPUT=SDD_CODE_INPUT, SDD_PREV=PREV_RESULTS)
setup = "/kaggle/working/sdd/scripts/kaggle_setup.sh"
if not os.path.exists(setup):
    setup = SDD_CODE_INPUT + "/scripts/kaggle_setup.sh"
!bash {setup}
%cd /kaggle/working/sdd

In [ ]:
import subprocess, sys
from pathlib import Path
import pandas as pd
import torch
from IPython.display import Image, Markdown, display

R = Path("/kaggle/working/results")
LOGS = Path("/kaggle/working/logs"); LOGS.mkdir(exist_ok=True)
N_GPU = max(torch.cuda.device_count(), 1)
print(f"GPUs: {N_GPU}")


def hours_left():
    return SESSION_HOURS - (time.time() - T0) / 3600


def sh(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True)
    if r.returncode:
        print(f"!! exit code {r.returncode}")


def grid(name, extra=""):
    """Run an experiment grid on all GPUs (one process per GPU) within the remaining session budget."""
    left = hours_left() - 0.3
    if left <= 0:
        print(f"skip {name}: session budget used up - re-run with PREV_RESULTS set")
        return
    if SMOKE:
        extra += " --smoke --datasets neu"
    dev = lambda g: g if torch.cuda.is_available() else "cpu"  # noqa: E731
    procs = []
    for g in range(N_GPU):
        log = LOGS / f"{name}_gpu{g}.log"
        cmd = (f"{sys.executable} scripts/run_grid.py {name} --device {dev(g)} --shard {g}/{N_GPU} "
               f"--max-hours {left:.2f} {extra}")
        print("$", cmd, ">", log)
        procs.append(subprocess.Popen(cmd, shell=True, stdout=open(log, "w"), stderr=subprocess.STDOUT))
    for p in procs:
        p.wait()
    for g in range(N_GPU):
        print(f"--- {name} gpu{g} (last lines of the log)")
        print("".join(open(LOGS / f"{name}_gpu{g}.log").readlines()[-12:]))
    sh(f"{sys.executable} scripts/run_grid.py {name} --status" + (" --smoke" if SMOKE else ""))


def show(pattern, width=720):
    for p in sorted(R.glob(pattern)):
        display(Markdown(f"**{p.relative_to(R)}**"))
        display(Image(filename=str(p), width=width))


def table(name):
    p = R / "tables" / f"{name}.md"
    display(Markdown(p.read_text(encoding="utf-8") if p.exists() else f"_{name}: no results yet_"))

## 3. Data
Converts every dataset to YOLO + COCO json with the fixed splits in `splits/`. **Look at the drawn labels before
trusting any training** (plan §8). If the splits were created in this session (first run ever), download
`/kaggle/working/sdd/splits/` and commit it, so every later session uses the same split.

In [ ]:
if RUN["data"]:
    sh(f"python scripts/prepare_data.py --datasets {' '.join(SOURCE_DATASETS + [TARGET_DATASET])} --n-vis 12")
    show("figures/labels/*/train_*.jpg", width=320)

## 4. EDA (plan §2.4)

In [ ]:
if RUN["eda"]:
    sh(f"python scripts/eda.py --datasets {' '.join(SOURCE_DATASETS)}")
    display(pd.read_csv(R / "eda/datasets_summary.csv").set_index("dataset").T)
    for ds in SOURCE_DATASETS:
        show(f"eda/{ds}/*.png")
        display(Markdown(f"**{ds}**: test boxes per candidate relative-size bins - choose bins with >= ~50 boxes "
                         "per group, then set `eval.rel_bins` in `configs/protocol.yaml`"))
        display(pd.read_csv(R / f"eda/{ds}/rel_bin_candidates.csv"))

## 5. Phase 1 - baselines + small improvement (plan §4)
YOLO11n, YOLO11n-P2, Faster R-CNN R50-FPN v2, RT-DETR-l on the source datasets (`configs/experiments/p1.yaml`).

In [ ]:
if RUN["p1"]:
    grid("p1")

## 6. Phase 2 (plan §5)
**6a. Screening** - each candidate module alone, 1 seed. With the phase-1 TIDE diagnosis (decision table §5.1),
use it to set A2/A3 in `configs/variants.yaml` before the ablation.

In [ ]:
if RUN["p2_screen"]:
    grid("p2_candidates")

**6b. Large images (PCB, GC10)** - SAHI-style sliced inference on the phase-1 checkpoints (no training) and
training on 640 tiles; scored on the same full test images as phase 1.

In [ ]:
if RUN["p2_tiling"]:
    grid("p2_tiling")

**6c. Ablation A0-A7** - A0, A1-A3, A7 with 3 seeds first; A4-A6 with 1 seed if time allows.

In [ ]:
if RUN["p2_ablation"]:
    grid("p2_ablation")
    grid("p2_ablation_rest")

## 7. Phase 3 - transfer learning to the target set (plan §6)
Source models (T2 = base, T3 = improved) on the merged source data, then T1/T2/T3 x {10, 25, 50, 100}% x
{full, freeze} x 3 seeds on the target set.

In [ ]:
if RUN["p3"]:
    sh(f"python scripts/prepare_data.py --merge {' '.join(SOURCE_DATASETS)} --name merged --n-vis 0")
    grid("p3_source")
    grid("p3_transfer")

## 8. Report - tables, figures, feature maps, error galleries

In [ ]:
if RUN["report"]:
    SM = " --include-smoke" if SMOKE else ""
    sh("python scripts/make_report.py" + SM)
    master = R / "master_results.csv"
    runs = pd.read_csv(master) if master.exists() and master.stat().st_size > 1 else pd.DataFrame()
    done = set(runs.run) if len(runs) else set()
    # error galleries for the phase-1 YOLO11n runs
    gal = sorted(r for r in done if r.removeprefix("smoke_").startswith("p1_")
                 and r.endswith("_yolo11n_base_s0"))
    if gal:
        sh("python scripts/make_report.py" + SM + " --gallery " + " ".join(gal))
    # feature maps: screened backbones if available, else phase-1 base vs P2
    for ds in SOURCE_DATASETS:
        pre = "smoke_" if SMOKE else ""
        fm = [r for r in (f"{pre}p2s_{ds}_yolo11n_{v}_s0" for v in ("base", "simam", "spd")) if r in done]
        fm = fm or [r for r in (f"{pre}p1_{ds}_yolo11n_base_s0", f"{pre}p1_{ds}_yolo11n_p2_s0") if r in done]
        if fm:
            sh(f"python scripts/feature_maps.py --dataset {ds} --n-images 2 --runs {' '.join(fm)}")

In [ ]:
if RUN["report"]:
    for t in ["p1_results", "p2s_results", "p2t_results", "p2_ablation", "p3_transfer"]:
        display(Markdown(f"### {t}"))
        table(t)
    show("figures/p1/*.png"); show("figures/p2*/*.png"); show("figures/p3/*.png")
    show("figures/feature_maps/*/compare_*.png", width=900)
    show("figures/errors/*/*.jpg", width=900)

In [ ]:
# slide material in one small zip (full runs incl. weights stay in /kaggle/working/results for the next session)
!cd /kaggle/working && zip -qr report_bundle.zip results/master_results.csv results/tables results/figures results/eda 2>/dev/null; ls -lh /kaggle/working/*.zip
print(f"session time used: {(time.time() - T0) / 3600:.2f} h")